# Lightweight Function Name Prediction from Metadata

**Objective:** Predict camelCase function names from function metadata using a lightweight ML model.

**Dataset:** Synthetic — 180 rows, 60 unique functions, 15 real-world domains.  
Each function has 3 description variants so the model learns to generalize across different phrasings.  
Format matches the assignment spec exactly: typed parameters, camelCase names, clean library names.

**Model:** TF-IDF + linear classifier — sub-millisecond inference, <1 MB total, suitable for Android.

---
**Sections:** 1. Dataset &nbsp;|&nbsp; 2. Preprocessing &nbsp;|&nbsp; 3. Training &nbsp;|&nbsp; 4. Evaluation &nbsp;|&nbsp; 5. Inference Demo &nbsp;|&nbsp; 6. Export & Deployment

## Section 1 — Dataset

180 hand-crafted rows across **15 domains**: MathUtils, StringUtils, NetworkService, AuthService, FileManager, DatabaseHelper, CryptoUtils, CacheManager, Logger, Validator, NotificationService, DateTimeUtils, ConversionUtils, DeviceUtils, ListUtils.  
Each of the **60 unique functions** has **3 description variants** — same metadata, different phrasing — so the model learns semantic generalization, not just memorization.

In [ ]:
import io
import re
import time
import os
import warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings('ignore')

from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

print('All libraries imported.')

In [ ]:
# ── Embedded dataset — no internet required ──────────────────────────────────
CSV_DATA = """\
description,parameters,return_type,library,keywords,param_count,function_name
Adds two integers and returns their sum,"int a, int b",int,MathUtils,"add,sum,arithmetic",2,addNumbers
Returns the sum of two given integer values,"int a, int b",int,MathUtils,"add,sum,arithmetic",2,addNumbers
Computes the total of two integers and returns the result,"int a, int b",int,MathUtils,"add,sum,arithmetic",2,addNumbers
Calculates the factorial of a non-negative integer,int n,long,MathUtils,"factorial,recursive,math",1,calculateFactorial
Returns the factorial value for a given non-negative number,int n,long,MathUtils,"factorial,recursive,math",1,calculateFactorial
Computes n factorial for the given integer n,int n,long,MathUtils,"factorial,recursive,math",1,calculateFactorial
Calculates the square root of a given number,double value,double,MathUtils,"sqrt,root,math",1,calculateSquareRoot
Returns the square root of the given double value,double value,double,MathUtils,"sqrt,root,math",1,calculateSquareRoot
Computes the mathematical square root of a value,double value,double,MathUtils,"sqrt,root,math",1,calculateSquareRoot
Returns the maximum of two integer values,"int a, int b",int,MathUtils,"max,maximum,compare",2,findMaximum
Finds and returns the larger of two given integers,"int a, int b",int,MathUtils,"max,maximum,compare",2,findMaximum
Compares two integers and returns the maximum one,"int a, int b",int,MathUtils,"max,maximum,compare",2,findMaximum
Raises base to the given exponent and returns the result,"double base, int exponent",double,MathUtils,"power,exponent,math",2,calculatePower
Computes base raised to the power of exponent,"double base, int exponent",double,MathUtils,"power,exponent,math",2,calculatePower
Returns the result of raising a base to an integer exponent,"double base, int exponent",double,MathUtils,"power,exponent,math",2,calculatePower
Reverses the characters in a given string,String input,String,StringUtils,"reverse,string,characters",1,reverseString
Returns the input string with its characters in reverse order,String input,String,StringUtils,"reverse,string,characters",1,reverseString
Flips the character order of a given string value,String input,String,StringUtils,"reverse,string,characters",1,reverseString
Converts all characters of a string to uppercase,String input,String,StringUtils,"uppercase,convert,string",1,convertToUpperCase
Returns the input string converted to all uppercase letters,String input,String,StringUtils,"uppercase,convert,string",1,convertToUpperCase
Transforms every character in the string to its uppercase form,String input,String,StringUtils,"uppercase,convert,string",1,convertToUpperCase
Checks if a given string is a palindrome,String input,boolean,StringUtils,"palindrome,check,string",1,checkPalindrome
Returns true if the input string reads the same forwards and backwards,String input,boolean,StringUtils,"palindrome,check,string",1,checkPalindrome
Determines whether a given string is a palindrome,String input,boolean,StringUtils,"palindrome,check,string",1,checkPalindrome
Splits a string into an array using the given delimiter,"String input, String delimiter",String[],StringUtils,"split,delimiter,array",2,splitByDelimiter
Divides the input string into parts based on the specified delimiter,"String input, String delimiter",String[],StringUtils,"split,delimiter,array",2,splitByDelimiter
Returns an array of substrings by splitting on the given delimiter,"String input, String delimiter",String[],StringUtils,"split,delimiter,array",2,splitByDelimiter
Removes leading and trailing whitespace from a string,String input,String,StringUtils,"trim,whitespace,string",1,trimWhitespace
Strips all leading and trailing spaces from the input string,String input,String,StringUtils,"trim,whitespace,string",1,trimWhitespace
Returns the input string with whitespace removed from both ends,String input,String,StringUtils,"trim,whitespace,string",1,trimWhitespace
Sends an HTTP GET request to the given URL,String url,HttpResponse,NetworkService,"http,get,request,api",1,sendGetRequest
Performs an HTTP GET call to the specified URL and returns the response,String url,HttpResponse,NetworkService,"http,get,request,api",1,sendGetRequest
Makes a GET request to the given URL and returns an HttpResponse,String url,HttpResponse,NetworkService,"http,get,request,api",1,sendGetRequest
Sends an HTTP POST request with a JSON body,"String url, String body",HttpResponse,NetworkService,"http,post,request,api",2,sendPostRequest
Performs an HTTP POST call with a body payload to a given URL,"String url, String body",HttpResponse,NetworkService,"http,post,request,api",2,sendPostRequest
Makes a POST request with the given body to the specified URL,"String url, String body",HttpResponse,NetworkService,"http,post,request,api",2,sendPostRequest
Downloads a file from the given URL to local storage,"String url, String path",boolean,NetworkService,"download,file,network",2,downloadFile
Fetches a remote file from a URL and saves it to the specified local path,"String url, String path",boolean,NetworkService,"download,file,network",2,downloadFile
Retrieves a file from the given URL and stores it at the given path,"String url, String path",boolean,NetworkService,"download,file,network",2,downloadFile
Checks whether the device has an active internet connection,,boolean,NetworkService,"connection,internet,network",0,checkConnectionStatus
Returns true if the device is currently connected to the internet,,boolean,NetworkService,"connection,internet,network",0,checkConnectionStatus
Determines if an active network connection is available on the device,,boolean,NetworkService,"connection,internet,network",0,checkConnectionStatus
Retrieves the HTTP status code from a response,HttpResponse response,int,NetworkService,"status,code,response",1,getResponseStatusCode
Returns the integer HTTP status code from an HttpResponse object,HttpResponse response,int,NetworkService,"status,code,response",1,getResponseStatusCode
Extracts the status code from the given HTTP response,HttpResponse response,int,NetworkService,"status,code,response",1,getResponseStatusCode
Generates a JWT token for a given user ID,String userId,String,AuthService,"jwt,token,generate,auth",1,generateJwtToken
Creates and returns a signed JWT authentication token for the user,String userId,String,AuthService,"jwt,token,generate,auth",1,generateJwtToken
Produces a new JWT token string for the specified user ID,String userId,String,AuthService,"jwt,token,generate,auth",1,generateJwtToken
Hashes a plain text password using bcrypt,String password,String,AuthService,"hash,password,bcrypt,security",1,hashPassword
Takes a plain password and returns its bcrypt hash,String password,String,AuthService,"hash,password,bcrypt,security",1,hashPassword
Encrypts a password string using the bcrypt hashing algorithm,String password,String,AuthService,"hash,password,bcrypt,security",1,hashPassword
Verifies a plain text password against its stored hash,"String password, String hash",boolean,AuthService,"verify,password,hash,auth",2,verifyPassword
Checks whether the given password matches the stored bcrypt hash,"String password, String hash",boolean,AuthService,"verify,password,hash,auth",2,verifyPassword
Returns true if the plain password matches the provided hash,"String password, String hash",boolean,AuthService,"verify,password,hash,auth",2,verifyPassword
Generates a one-time password for two-factor authentication,String userId,String,AuthService,"otp,generate,twofactor,auth",1,generateOtpCode
Creates a new OTP code for the given user for 2FA verification,String userId,String,AuthService,"otp,generate,twofactor,auth",1,generateOtpCode
Produces a one-time numeric password for two-factor authentication,String userId,String,AuthService,"otp,generate,twofactor,auth",1,generateOtpCode
Creates a new user session and returns the session ID,String userId,String,AuthService,"session,create,user,auth",1,createUserSession
Initializes a new session for the specified user and returns its ID,String userId,String,AuthService,"session,create,user,auth",1,createUserSession
Starts a new authenticated session for a user and returns the session identifier,String userId,String,AuthService,"session,create,user,auth",1,createUserSession
Reads and returns the full contents of a file,String filePath,String,FileManager,"read,file,contents",1,readFileContents
Opens a file at the given path and returns its complete text content,String filePath,String,FileManager,"read,file,contents",1,readFileContents
Returns the entire content of the file located at the given path,String filePath,String,FileManager,"read,file,contents",1,readFileContents
Writes given content to a file at the specified path,"String filePath, String content",boolean,FileManager,"write,file,contents",2,writeFileContents
Saves the provided string content to a file at the specified path,"String filePath, String content",boolean,FileManager,"write,file,contents",2,writeFileContents
Creates or overwrites a file with the given content at the given path,"String filePath, String content",boolean,FileManager,"write,file,contents",2,writeFileContents
Deletes the file at the specified path,String filePath,boolean,FileManager,"delete,file,remove",1,deleteFile
Removes the file located at the given file path,String filePath,boolean,FileManager,"delete,file,remove",1,deleteFile
Permanently deletes the file found at the specified path,String filePath,boolean,FileManager,"delete,file,remove",1,deleteFile
Checks whether a file exists at the given path,String filePath,boolean,FileManager,"check,file,exists",1,checkFileExists
Returns true if a file is present at the specified file path,String filePath,boolean,FileManager,"check,file,exists",1,checkFileExists
Determines whether a file exists at the provided path,String filePath,boolean,FileManager,"check,file,exists",1,checkFileExists
Copies a file from source path to destination path,"String source, String destination",boolean,FileManager,"copy,file,duplicate",2,copyFile
Duplicates a file from the source location to the destination path,"String source, String destination",boolean,FileManager,"copy,file,duplicate",2,copyFile
Creates a copy of the file at source and places it at destination,"String source, String destination",boolean,FileManager,"copy,file,duplicate",2,copyFile
Inserts a new record into the specified database table,"String table, Map data",boolean,DatabaseHelper,"insert,record,database",2,insertRecord
Adds a new data entry to the given database table,"String table, Map data",boolean,DatabaseHelper,"insert,record,database",2,insertRecord
Creates a new row with the given data in the specified table,"String table, Map data",boolean,DatabaseHelper,"insert,record,database",2,insertRecord
Retrieves a record from the database by its ID,"String table, int id",Map,DatabaseHelper,"select,record,query",2,selectRecord
Fetches a single database record using the given table and ID,"String table, int id",Map,DatabaseHelper,"select,record,query",2,selectRecord
Returns the database row matching the given ID from the specified table,"String table, int id",Map,DatabaseHelper,"select,record,query",2,selectRecord
Updates an existing record in the database by ID,"String table, int id, Map data",boolean,DatabaseHelper,"update,record,database",3,updateRecord
Modifies the data of a database record identified by the given ID,"String table, int id, Map data",boolean,DatabaseHelper,"update,record,database",3,updateRecord
Changes the fields of an existing database row using the given ID,"String table, int id, Map data",boolean,DatabaseHelper,"update,record,database",3,updateRecord
Executes a raw SQL query and returns the results,String query,List,DatabaseHelper,"query,execute,sql",1,executeQuery
Runs the given SQL statement and returns the resulting dataset,String query,List,DatabaseHelper,"query,execute,sql",1,executeQuery
Processes a raw SQL query string and returns matching records,String query,List,DatabaseHelper,"query,execute,sql",1,executeQuery
Returns the total number of records in a table,String table,int,DatabaseHelper,"count,records,table",1,countRecords
Gets the count of all rows currently stored in the given table,String table,int,DatabaseHelper,"count,records,table",1,countRecords
Counts and returns the total number of entries in the specified table,String table,int,DatabaseHelper,"count,records,table",1,countRecords
Encrypts plaintext using the AES encryption algorithm,"String plaintext, String key",String,CryptoUtils,"encrypt,aes,crypto",2,encryptAes
Returns AES-encrypted ciphertext for the given plain text and key,"String plaintext, String key",String,CryptoUtils,"encrypt,aes,crypto",2,encryptAes
Applies AES encryption to the input text using the provided secret key,"String plaintext, String key",String,CryptoUtils,"encrypt,aes,crypto",2,encryptAes
Computes the SHA-256 hash of a given string,String input,String,CryptoUtils,"hash,sha256,crypto",1,hashSha256
Returns the SHA-256 digest of the given input string,String input,String,CryptoUtils,"hash,sha256,crypto",1,hashSha256
Generates a SHA-256 cryptographic hash for the given string value,String input,String,CryptoUtils,"hash,sha256,crypto",1,hashSha256
Encodes a string to Base64 format,String input,String,CryptoUtils,"encode,base64,crypto",1,encodeBase64
Converts the input string to its Base64 encoded representation,String input,String,CryptoUtils,"encode,base64,crypto",1,encodeBase64
Returns the Base64 encoding of the given input string,String input,String,CryptoUtils,"encode,base64,crypto",1,encodeBase64
Generates a cryptographically secure random key,int length,String,CryptoUtils,"random,key,generate,crypto",1,generateRandomKey
Creates a secure random key string of the specified length,int length,String,CryptoUtils,"random,key,generate,crypto",1,generateRandomKey
Returns a new cryptographically strong random key,int length,String,CryptoUtils,"random,key,generate,crypto",1,generateRandomKey
Verifies a digital signature against the original data,"String data, String signature, String publicKey",boolean,CryptoUtils,"verify,signature,crypto",3,verifySignature
Checks if the given signature is valid for the original data using a public key,"String data, String signature, String publicKey",boolean,CryptoUtils,"verify,signature,crypto",3,verifySignature
Returns true if the digital signature matches the data and public key,"String data, String signature, String publicKey",boolean,CryptoUtils,"verify,signature,crypto",3,verifySignature
Stores a key-value pair in the cache with an expiry time,"String key, Object value, int ttl",void,CacheManager,"cache,store,value,ttl",3,cacheValue
Saves a value in the cache under the given key with a TTL,"String key, Object value, int ttl",void,CacheManager,"cache,store,value,ttl",3,cacheValue
Puts a key-value entry into the cache with the specified time-to-live,"String key, Object value, int ttl",void,CacheManager,"cache,store,value,ttl",3,cacheValue
Retrieves a cached value by its key,String key,Object,CacheManager,"cache,retrieve,get",1,getCachedValue
Returns the value stored in cache for the given key,String key,Object,CacheManager,"cache,retrieve,get",1,getCachedValue
Looks up and returns the cached entry associated with the given key,String key,Object,CacheManager,"cache,retrieve,get",1,getCachedValue
Clears all entries from the cache,,void,CacheManager,"cache,clear,flush",0,clearAllCache
Removes every entry currently stored in the cache,,void,CacheManager,"cache,clear,flush",0,clearAllCache
Flushes and empties the entire cache store,,void,CacheManager,"cache,clear,flush",0,clearAllCache
Logs an informational message with a timestamp,String message,void,Logger,"log,info,message",1,logInfo
Records an info-level log message with the current timestamp,String message,void,Logger,"log,info,message",1,logInfo
Writes an informational message to the application log,String message,void,Logger,"log,info,message",1,logInfo
Logs an error message along with the exception details,"String message, Exception e",void,Logger,"log,error,exception",2,logError
Records an error and its associated exception in the log,"String message, Exception e",void,Logger,"log,error,exception",2,logError
Writes an error-level message and exception details to the log,"String message, Exception e",void,Logger,"log,error,exception",2,logError
Retrieves log history as a list of log entries,int limit,List,Logger,"log,history,retrieve",1,getLogHistory
Returns a list of the most recent log records up to the given limit,int limit,List,Logger,"log,history,retrieve",1,getLogHistory
Fetches the stored log history as a collection of entries,int limit,List,Logger,"log,history,retrieve",1,getLogHistory
Validates whether the given string is a valid email address,String email,boolean,Validator,"validate,email,format",1,validateEmail
Returns true if the input matches a valid email address format,String email,boolean,Validator,"validate,email,format",1,validateEmail
Checks whether the provided string conforms to email address rules,String email,boolean,Validator,"validate,email,format",1,validateEmail
Validates whether the given string is a valid phone number,String phone,boolean,Validator,"validate,phone,format",1,validatePhoneNumber
Returns true if the input is a properly formatted phone number,String phone,boolean,Validator,"validate,phone,format",1,validatePhoneNumber
Checks whether the provided string is a valid phone number,String phone,boolean,Validator,"validate,phone,format",1,validatePhoneNumber
Validates whether the given string is a valid URL,String url,boolean,Validator,"validate,url,format",1,validateUrl
Returns true if the input conforms to a valid URL format,String url,boolean,Validator,"validate,url,format",1,validateUrl
Checks whether the provided string is a well-formed URL,String url,boolean,Validator,"validate,url,format",1,validateUrl
Checks if the password meets the required strength criteria,String password,boolean,Validator,"validate,password,strength",1,validatePasswordStrength
Returns true if the password satisfies all strength requirements,String password,boolean,Validator,"validate,password,strength",1,validatePasswordStrength
Validates that the given password meets the minimum strength rules,String password,boolean,Validator,"validate,password,strength",1,validatePasswordStrength
Sends an email notification to the given recipient,"String email, String subject, String body",boolean,NotificationService,"email,notification,send",3,sendEmailNotification
Delivers an email message with subject and body to the specified address,"String email, String subject, String body",boolean,NotificationService,"email,notification,send",3,sendEmailNotification
Dispatches an email notification to the specified recipient,"String email, String subject, String body",boolean,NotificationService,"email,notification,send",3,sendEmailNotification
Sends a push notification to a device by its token,"String deviceToken, String message",boolean,NotificationService,"push,notification,send",2,sendPushNotification
Delivers a push message to a mobile device using its device token,"String deviceToken, String message",boolean,NotificationService,"push,notification,send",2,sendPushNotification
Dispatches a push notification to the specified device token,"String deviceToken, String message",boolean,NotificationService,"push,notification,send",2,sendPushNotification
Schedules a notification to be delivered at a given time,"String userId, String message, long timestamp",boolean,NotificationService,"schedule,notification,time",3,scheduleNotification
Queues a notification for delivery to a user at the specified timestamp,"String userId, String message, long timestamp",boolean,NotificationService,"schedule,notification,time",3,scheduleNotification
Sets up a notification to be sent to the user at a future timestamp,"String userId, String message, long timestamp",boolean,NotificationService,"schedule,notification,time",3,scheduleNotification
Returns the current date and time as a Unix timestamp,,long,DateTimeUtils,"timestamp,current,time",0,getCurrentTimestamp
Gets the current system time as a Unix epoch timestamp,,long,DateTimeUtils,"timestamp,current,time",0,getCurrentTimestamp
Retrieves the current date and time in Unix timestamp format,,long,DateTimeUtils,"timestamp,current,time",0,getCurrentTimestamp
Formats a Unix timestamp to a human-readable date string,"long timestamp, String format",String,DateTimeUtils,"format,date,string",2,formatDate
Converts a Unix timestamp into a formatted date string using the given pattern,"long timestamp, String format",String,DateTimeUtils,"format,date,string",2,formatDate
Returns a readable date string from a timestamp using the specified format,"long timestamp, String format",String,DateTimeUtils,"format,date,string",2,formatDate
Calculates the number of days between two date timestamps,"long startDate, long endDate",int,DateTimeUtils,"difference,days,date",2,calculateDateDifference
Returns the difference in days between two given Unix timestamps,"long startDate, long endDate",int,DateTimeUtils,"difference,days,date",2,calculateDateDifference
Computes how many days separate two given date timestamps,"long startDate, long endDate",int,DateTimeUtils,"difference,days,date",2,calculateDateDifference
Converts temperature from Celsius to Fahrenheit,float celsius,float,ConversionUtils,"convert,celsius,fahrenheit,temperature",1,convertCelsiusToFahrenheit
Returns the Fahrenheit equivalent of the given Celsius temperature,float celsius,float,ConversionUtils,"convert,celsius,fahrenheit,temperature",1,convertCelsiusToFahrenheit
Transforms a Celsius temperature value to its Fahrenheit equivalent,float celsius,float,ConversionUtils,"convert,celsius,fahrenheit,temperature",1,convertCelsiusToFahrenheit
Converts distance from kilometers to miles,double km,double,ConversionUtils,"convert,kilometers,miles,distance",1,convertKilometersToMiles
Returns the mile equivalent of the given distance in kilometers,double km,double,ConversionUtils,"convert,kilometers,miles,distance",1,convertKilometersToMiles
Transforms a kilometer distance value to its equivalent in miles,double km,double,ConversionUtils,"convert,kilometers,miles,distance",1,convertKilometersToMiles
Converts weight from kilograms to pounds,double kg,double,ConversionUtils,"convert,kilograms,pounds,weight",1,convertKgToPounds
Returns the pound equivalent of the given weight in kilograms,double kg,double,ConversionUtils,"convert,kilograms,pounds,weight",1,convertKgToPounds
Transforms a weight in kilograms to its equivalent value in pounds,double kg,double,ConversionUtils,"convert,kilograms,pounds,weight",1,convertKgToPounds
Returns the current battery level as a percentage,,int,DeviceUtils,"battery,level,device",0,getBatteryLevel
Gets the device battery percentage at the current time,,int,DeviceUtils,"battery,level,device",0,getBatteryLevel
Retrieves the current battery charge level of the device,,int,DeviceUtils,"battery,level,device",0,getBatteryLevel
Returns the unique identifier of the device,,String,DeviceUtils,"device,id,unique",0,getDeviceId
Gets the unique hardware identifier assigned to this device,,String,DeviceUtils,"device,id,unique",0,getDeviceId
Retrieves the device-specific unique ID string,,String,DeviceUtils,"device,id,unique",0,getDeviceId
Returns the type of network the device is connected to,,String,DeviceUtils,"network,type,wifi,device",0,getNetworkType
Gets the current network connection type such as WiFi or mobile,,String,DeviceUtils,"network,type,wifi,device",0,getNetworkType
Retrieves the active network connection type for the device,,String,DeviceUtils,"network,type,wifi,device",0,getNetworkType
Sorts a list of integers in ascending order,List items,List,ListUtils,"sort,ascending,list",1,sortListAscending
Returns the given list of integers sorted from smallest to largest,List items,List,ListUtils,"sort,ascending,list",1,sortListAscending
Arranges a list of integers in ascending numerical order,List items,List,ListUtils,"sort,ascending,list",1,sortListAscending
Removes duplicate elements from a list,List items,List,ListUtils,"filter,duplicates,unique",1,filterDuplicates
Returns a new list with all duplicate values removed,List items,List,ListUtils,"filter,duplicates,unique",1,filterDuplicates
Eliminates repeated elements from the given list,List items,List,ListUtils,"filter,duplicates,unique",1,filterDuplicates
Merges two lists into a single combined list,"List first, List second",List,ListUtils,"merge,combine,list",2,mergeLists
Combines two input lists into one unified list,"List first, List second",List,ListUtils,"merge,combine,list",2,mergeLists
Returns a single list formed by joining two given lists together,"List first, List second",List,ListUtils,"merge,combine,list",2,mergeLists
"""

df = pd.read_csv(io.StringIO(CSV_DATA))
df['parameters'] = df['parameters'].fillna('')

print(f'Rows          : {len(df)}')
print(f'Unique funcs  : {df["function_name"].nunique()}')
print(f'Domains       : {df["library"].nunique()}')
print(f'Variants/func : {len(df) // df["function_name"].nunique()}')
print(f'\nDomain list: {sorted(df["library"].unique())}')
print('\nSample rows:')
df.head(6)

In [ ]:
# Save CSV to disk (upload this to GitHub)
df.to_csv('function_metadata.csv', index=False)
print("function_metadata.csv saved.")
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nParam count distribution:\n{df["param_count"].value_counts().sort_index()}')

## Section 2 — Preprocessing & Feature Engineering

All metadata columns are merged into a single enriched text string per row.  
- **Keywords** are repeated **3×** to boost their TF-IDF weight  
- **CamelCase** library names are tokenized (`MathUtils` → `math utils`)  
- **TF-IDF** with unigrams + bigrams creates a sparse feature matrix

In [ ]:
def camel_tokenize(text):
    """Split CamelCase into lowercase words: MathUtils -> math utils"""
    return re.sub(r'([A-Z])', r' \1', str(text)).lower().strip()

def build_feature_text(row):
    """
    Combine metadata columns into one enriched string.
    Keywords repeated 3x for stronger TF-IDF signal.
    """
    desc   = str(row['description']).lower()
    params = str(row['parameters']).lower()
    ret    = camel_tokenize(row['return_type'])
    lib    = camel_tokenize(row['library'])         # e.g. MathUtils -> math utils
    kw     = str(row['keywords']).replace(',', ' ').lower()
    pc     = f"params_{int(row['param_count'])}"
    return f"{desc} {params} {ret} {lib} {kw} {kw} {kw} {pc}"

df['feature_text'] = df.apply(build_feature_text, axis=1)

print('Feature text examples:')
for fn in ['addNumbers', 'sendGetRequest', 'validateEmail']:
    sample = df[df['function_name'] == fn].iloc[0]
    print(f'  [{fn}]:')
    print(f'  {sample["feature_text"]}')
    print()

In [ ]:
# Label encode the target column
le = LabelEncoder()
y  = le.fit_transform(df['function_name'])
print(f'Classes: {len(le.classes_)} unique function names')

# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # unigrams + bigrams
    max_features=3000,     # cap vocabulary for small model size
    sublinear_tf=True,     # log(1+tf) to dampen high-frequency terms
    min_df=1               # include all terms (small dataset)
)
X = vectorizer.fit_transform(df['feature_text'])
print(f'Feature matrix : {X.shape[0]} rows x {X.shape[1]} features')
print(f'Sparsity       : {100*(1 - X.nnz/(X.shape[0]*X.shape[1])):.1f}%')

# Stratified split: 120 train (2 per class) / 60 test (1 per class)
# stratify=y ensures every class appears in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=60, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples ({X_train.shape[0]//len(le.classes_)} per class) '
      f'| Test: {X_test.shape[0]} samples (1 per class)')

## Section 3 — Model Training

Two lightweight linear classifiers are trained and compared:
- **Logistic Regression** — probability outputs enable top-k predictions  
- **LinearSVC** — fastest inference, no probability estimates  

Both are ideal for high-dimensional sparse TF-IDF features and run in sub-millisecond time.

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(
        C=2.0, max_iter=1000, solver='lbfgs',
        class_weight='balanced', random_state=42
    ),
    'LinearSVC': LinearSVC(
        C=1.5, max_iter=2000,
        class_weight='balanced', random_state=42
    )
}

results = {}
cv = ShuffleSplit(n_splits=5, test_size=60, random_state=42)

print(f'{"Model":<22} {"Test Acc":>10} {"CV Mean":>10} {"CV Std":>8} {"Train ms":>10} {"Infer ms":>10}')
print('-' * 78)

for name, model in models.items():
    # Train
    t0 = time.time()
    model.fit(X_train, y_train)
    train_ms = (time.time() - t0) * 1000

    # Test accuracy
    y_pred   = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)

    # 5-fold cross-validation on full dataset
    cv_scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

    # Inference latency — avg over 500 single-sample predictions
    t1 = time.time()
    for _ in range(500):
        model.predict(X_test[0])
    infer_ms = (time.time() - t1) / 500 * 1000

    results[name] = {
        'model':    model,
        'test_acc': test_acc,
        'cv_mean':  cv_scores.mean(),
        'cv_std':   cv_scores.std(),
        'train_ms': train_ms,
        'infer_ms': infer_ms
    }
    print(f'{name:<22} {test_acc:>10.4f} {cv_scores.mean():>10.4f} '
          f'{cv_scores.std():>8.4f} {train_ms:>10.1f} {infer_ms:>10.4f}')

## Section 4 — Evaluation

Select the best model by test accuracy.  
Measure **top-3 accuracy** (how often the correct name appears in the top 3 predictions)  
and run a **500-sample speed benchmark**.

In [ ]:
# Select best model
best_name  = max(results, key=lambda k: results[k]['test_acc'])
best_model = results[best_name]['model']
lr_model   = results['LogisticRegression']['model']

print(f'Best model     : {best_name}')
print(f'Test accuracy  : {results[best_name]["test_acc"]:.4f}')
print(f'CV accuracy    : {results[best_name]["cv_mean"]:.4f} +/- {results[best_name]["cv_std"]:.4f}')

# Top-3 accuracy via predict_proba (Logistic Regression)
proba    = lr_model.predict_proba(X_test)
top3_acc = sum(
    1 for i, true in enumerate(y_test)
    if true in np.argsort(proba[i])[::-1][:3]
) / len(y_test)
print(f'Top-3 accuracy : {top3_acc:.4f}  (LogisticRegression)')

# Speed benchmark
latencies = []
for _ in range(500):
    t = time.time()
    best_model.predict(X_test[0])
    latencies.append((time.time() - t) * 1000)
lat = np.array(latencies)

print(f'\nInference benchmark (500 runs):')
print(f'  Mean       : {lat.mean():.4f} ms')
print(f'  Median     : {np.median(lat):.4f} ms')
print(f'  Min        : {lat.min():.4f} ms')
print(f'  Throughput : {1000/lat.mean():.0f} predictions/sec')

In [ ]:
# Full classification report on test set
y_pred_test  = best_model.predict(X_test)
test_labels  = sorted(set(y_test))
target_names = le.inverse_transform(test_labels)

print(f'Classification Report ({best_name}):\n')
print(classification_report(
    y_test, y_pred_test,
    labels=test_labels,
    target_names=target_names,
    zero_division=0
))

## Section 5 — Inference Demo

`predict_function_name()` takes a **raw combined metadata string** — exactly as described in the assignment —  
and returns the predicted camelCase function name, top-3 candidates with confidence scores, and latency.

In [ ]:
def predict_function_name(raw_input, top_k=3):
    """
    Predict a camelCase function name from a raw metadata string.

    Args:
        raw_input (str): Combined metadata, e.g.
            'Adds two integers int a int b return int keywords add sum'
        top_k (int): Number of top candidates to return (default 3)

    Returns:
        dict:
            prediction (str)   — top-1 predicted function name
            top_k      (list)  — [(name, probability), ...]
            latency_ms (float) — inference time in milliseconds
    """
    cleaned = raw_input.strip().lower()

    t0  = time.time()
    vec = vectorizer.transform([cleaned])
    idx = best_model.predict(vec)[0]
    prediction = le.inverse_transform([idx])[0]
    latency_ms = (time.time() - t0) * 1000

    # Top-k probabilities via Logistic Regression
    probs    = lr_model.predict_proba(vec)[0]
    top_idxs = np.argsort(probs)[::-1][:top_k]
    top_list = [
        (le.inverse_transform([i])[0], round(float(probs[i]), 4))
        for i in top_idxs
    ]

    return {
        'prediction': prediction,
        'top_k':      top_list,
        'latency_ms': round(latency_ms, 4)
    }

print('predict_function_name() ready.')

In [ ]:
# ── Assignment inference examples ─────────────────────────────────────────────
examples = [
    {
        'label':    'Assignment example 1',
        'input':    'Adds two integers int a int b return int keywords add sum',
        'expected': 'addNumbers'
    },
    {
        'label':    'Assignment example 2',
        'input':    'Converts temperature from Celsius to Fahrenheit float celsius return float keywords convert temperature celsius fahrenheit',
        'expected': 'convertCelsiusToFahrenheit'
    },
    {
        'label':    'Assignment example 3',
        'input':    'Sends HTTP GET request to given URL String url return HttpResponse keywords http get request api',
        'expected': 'sendGetRequest'
    },
    {
        'label':    'Extra example 4',
        'input':    'Validates whether the given string is a valid email address String email return boolean keywords validate email format',
        'expected': 'validateEmail'
    },
    {
        'label':    'Extra example 5',
        'input':    'Generates a JWT token for a given user ID String userId return String keywords jwt token generate auth',
        'expected': 'generateJwtToken'
    },
]

print('=' * 75)
print(f'{"Example":<22} {"Prediction":<32} {"Match?":<8} {"ms":>5}')
print('=' * 75)
for ex in examples:
    r = predict_function_name(ex['input'])
    match = 'YES' if r['prediction'] == ex['expected'] else 'NO'
    print(f"{ex['label']:<22} {r['prediction']:<32} {match:<8} {r['latency_ms']:>5.3f}")
    print(f"  Top-3 : {r['top_k']}")
    print(f"  Input : {ex['input'][:70]}..." if len(ex['input'])>70 else f"  Input : {ex['input']}")
    print()

## Section 6 — Export & Deployment Demo

The trained model, TF-IDF vectorizer, and label encoder are exported as `.pkl` files.  
Total size is verified to be **under 2 MB** (Android deployment limit).  
The artifacts are reloaded from disk and used for a fresh end-to-end prediction to confirm the deployment pipeline works.

In [ ]:
# Export artifacts
joblib.dump(best_model, 'fn_predictor.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
joblib.dump(le,         'label_encoder.pkl')

# Report file sizes
artifacts = ['fn_predictor.pkl', 'vectorizer.pkl', 'label_encoder.pkl']
total_kb  = 0
print(f'{"Artifact":<25} {"Size":>10}')
print('-' * 38)
for f in artifacts:
    kb = os.path.getsize(f) / 1024
    total_kb += kb
    print(f'{f:<25} {kb:>8.1f} KB')
print('-' * 38)
print(f'{"TOTAL":<25} {total_kb:>8.1f} KB')

assert total_kb < 2048, f'Package too large: {total_kb:.1f} KB (limit: 2048 KB)'
print(f'\nSize check PASSED — {total_kb:.1f} KB < 2048 KB (Android deployment limit)')

In [ ]:
# ── Reload from disk and verify end-to-end ──────────────────────────────────
loaded_model = joblib.load('fn_predictor.pkl')
loaded_vec   = joblib.load('vectorizer.pkl')
loaded_le    = joblib.load('label_encoder.pkl')

def predict_from_disk(raw_input):
    """End-to-end inference using only reloaded artifacts (simulates production)."""
    cleaned = raw_input.strip().lower()
    t0   = time.time()
    vec  = loaded_vec.transform([cleaned])
    idx  = loaded_model.predict(vec)[0]
    name = loaded_le.inverse_transform([idx])[0]
    ms   = (time.time() - t0) * 1000
    return name, round(ms, 4)

# Use the assignment's own example
test_input = 'Adds two integers int a int b return int keywords add sum'
pred_name, pred_ms = predict_from_disk(test_input)

print('=' * 55)
print('        DEPLOYMENT VERIFICATION')
print('=' * 55)
print(f'Input      : {test_input}')
print(f'Prediction : {pred_name}')
print(f'Latency    : {pred_ms} ms')
print('=' * 55)
print('Artifacts reloaded and verified successfully.')

In [ ]:
# ── Final Summary ────────────────────────────────────────────────────────────
print('=' * 55)
print('            PROJECT SUMMARY')
print('=' * 55)
print(f'Dataset          : Synthetic (hand-crafted)')
print(f'Total rows       : {len(df)}')
print(f'Unique functions : {df["function_name"].nunique()}')
print(f'Domains covered  : {df["library"].nunique()}')
print(f'Best model       : {best_name}')
print(f'Test accuracy    : {results[best_name]["test_acc"]:.4f}')
print(f'Top-3 accuracy   : {top3_acc:.4f}')
print(f'CV accuracy      : {results[best_name]["cv_mean"]:.4f} +/- {results[best_name]["cv_std"]:.4f}')
print(f'Inference latency: {results[best_name]["infer_ms"]:.4f} ms/sample')
print(f'Model pkg size   : {total_kb:.1f} KB')
print('=' * 55)